# V2 — A2 با آموزش چندموقعیتی رخداد

مدل A2 همان ResNet18 فریز‌شده با mean-max pooling است، اما train اکنون سه موقعیت زمانی برای رخداد دارد. validation همان ۱۲۰ ویدئوی ثابت V2-W2 است. ویژگی‌های validation از cache معتبر قبلی reuse می‌شوند و فقط ویژگی‌های ۱٬۴۴۰ sequence جدید train استخراج می‌شوند.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import random

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from tqdm.auto import tqdm

DATA_ROOT = Path(r'P:\\NexarCollisionData')
MULTIPOS_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2_multipos.csv'
FRAME_INDEX_PATH = DATA_ROOT / 'frame_cache_index_v2_multipos.csv'
BASE_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2.csv'
BASE_FEATURE_CACHE_PATH = DATA_ROOT / 'processed_v2' / 'resnet18_imagenet_features_v2_w2_16x224x320.pt'

FEATURE_CACHE_PATH = DATA_ROOT / 'processed_v2' / 'resnet18_imagenet_features_v2_multipos_16x224x320.pt'
TRAIN_FEATURE_PARTIAL_PATH = DATA_ROOT / 'processed_v2' / 'resnet18_imagenet_features_v2_multipos_train_partial.pt'
MODEL_DIR = DATA_ROOT / 'models_v2'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'resnet18_meanmax_pooling_frozen_multipos'
BASELINE_METRICS_PATH = MODEL_DIR / 'resnet18_meanmax_pooling_frozen_metrics.json'
NUM_FRAMES = 16
FEATURE_DIM = 512
FRAME_BATCH_SIZE = 2
HEAD_BATCH_SIZE = 64
EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MINIMUM_ACCIDENT_RECALL = 0.80
SEED = 42

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
PREPROCESSING_VERSION = 'v2_multipos_rgb_letterbox_replicate_224x320'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

assert MULTIPOS_MANIFEST_PATH.exists(), 'Run notebook 18 first.'
assert FRAME_INDEX_PATH.exists(), 'Run notebook 19 first.'
assert BASE_MANIFEST_PATH.exists() and BASE_FEATURE_CACHE_PATH.exists(), 'The frozen V2 feature cache is required for validation reuse.'
print(f'Device: {device}')

Device: cpu


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
sequence_manifest = pd.read_csv(MULTIPOS_MANIFEST_PATH).copy()
frame_index = pd.read_csv(FRAME_INDEX_PATH).copy()
base_manifest = pd.read_csv(BASE_MANIFEST_PATH).copy()

for table in (sequence_manifest, frame_index, base_manifest):
    table['video_id'] = table['video_id'].astype(str)
    table['label'] = table['label'].astype(int)
frame_index['frame_index'] = frame_index['frame_index'].astype(int)
frame_index['frame_valid'] = frame_index['frame_valid'].astype(str).str.lower().eq('true')

sequence_manifest = sequence_manifest.sort_values(
    ['split', 'video_id', 'sequence_variant'],
    key=lambda values: values.astype(int) if values.name == 'video_id' else values,
).reset_index(drop=True)
assert len(sequence_manifest) == 1560
assert sequence_manifest['sequence_id'].is_unique
assert sequence_manifest.groupby(['split', 'label']).size().to_dict() == {
    ('train', 0): 720, ('train', 1): 720, ('validation', 0): 60, ('validation', 1): 60,
}
assert len(frame_index) == len(sequence_manifest) * NUM_FRAMES
assert frame_index['frame_valid'].all(), 'All cached frames must be valid.'
assert frame_index['frame_path'].map(lambda value: Path(value).is_file()).all(), 'A cached frame is missing.'
assert frame_index.groupby('sequence_id').size().eq(NUM_FRAMES).all()
assert set(frame_index['sequence_id']) == set(sequence_manifest['sequence_id'])

train_sequences = sequence_manifest.loc[sequence_manifest['split'].eq('train')].copy()
validation_sequences = sequence_manifest.loc[sequence_manifest['split'].eq('validation')].copy()
assert len(train_sequences) == 1440 and len(validation_sequences) == 120
train_frame_groups = {
    sequence_id: group.sort_values('frame_index').reset_index(drop=True)
    for sequence_id, group in frame_index.loc[frame_index['split'].eq('train')].groupby('sequence_id', sort=False)
}
assert set(train_sequences['sequence_id']) == set(train_frame_groups)

print('Fixed split with expanded train sequences:')
display(pd.crosstab(sequence_manifest['split'], sequence_manifest['label']))

Fixed split with expanded train sequences:


label,0,1
split,,
train,720,720
validation,60,60


In [3]:
class TrainSequenceFrameDataset(Dataset):
    def __init__(self, sequence_table: pd.DataFrame, grouped_frames: dict[str, pd.DataFrame]):
        self.sequence_table = sequence_table.reset_index(drop=True)
        self.grouped_frames = grouped_frames

    def __len__(self) -> int:
        return len(self.sequence_table)

    @staticmethod
    def load_and_normalize_rgb(path: str) -> torch.Tensor:
        frame_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if frame_bgr is None:
            raise RuntimeError(f'Cannot read cached frame: {path}')
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(frame_rgb.copy()).permute(2, 0, 1).float().div_(255.0)
        return (tensor - IMAGENET_MEAN) / IMAGENET_STD

    def __getitem__(self, index: int):
        sequence_row = self.sequence_table.iloc[index]
        frames = self.grouped_frames[sequence_row.sequence_id]
        assert len(frames) == NUM_FRAMES
        images = torch.stack([self.load_and_normalize_rgb(path) for path in frames['frame_path']])
        return images, int(sequence_row.label), sequence_row.sequence_id

example_images, example_label, example_sequence_id = TrainSequenceFrameDataset(train_sequences, train_frame_groups)[0]
assert example_images.shape == (NUM_FRAMES, 3, 224, 320)
print({'images_shape': tuple(example_images.shape), 'label': example_label, 'sequence_id': example_sequence_id})

{'images_shape': (16, 3, 224, 320), 'label': 1, 'sequence_id': 'V2-MP-3positions_0_p0'}


In [4]:
expected_sequence_ids = sequence_manifest['sequence_id'].tolist()
feature_payload = None
reused_complete_feature_cache = False

if FEATURE_CACHE_PATH.exists():
    candidate = torch.load(FEATURE_CACHE_PATH, map_location='cpu', weights_only=False)
    if (
        candidate.get('sequence_ids') == expected_sequence_ids
        and tuple(candidate.get('features', torch.empty(0)).shape) == (1560, NUM_FRAMES, FEATURE_DIM)
        and candidate.get('preprocessing_version') == PREPROCESSING_VERSION
    ):
        feature_payload = candidate
        reused_complete_feature_cache = True

if not reused_complete_feature_cache:
    # The frozen V2-W2 validation features are exactly the same validation clips.
    base_feature_payload = torch.load(BASE_FEATURE_CACHE_PATH, map_location='cpu', weights_only=False)
    assert tuple(base_feature_payload['features'].shape) == (600, NUM_FRAMES, FEATURE_DIM)
    assert base_feature_payload.get('preprocessing_version') == 'v2_w2_rgb_letterbox_replicate_224x320'
    base_feature_by_sequence = {
        sequence_id: base_feature_payload['features'][index].float().cpu()
        for index, sequence_id in enumerate(base_feature_payload['sequence_ids'])
    }
    base_sequence_id_by_video = base_manifest.set_index('video_id')['sequence_id'].to_dict()

    partial_features: dict[str, torch.Tensor] = {}
    if TRAIN_FEATURE_PARTIAL_PATH.exists():
        partial_payload = torch.load(TRAIN_FEATURE_PARTIAL_PATH, map_location='cpu', weights_only=False)
        if partial_payload.get('preprocessing_version') == PREPROCESSING_VERSION:
            candidate_features = partial_payload.get('features_by_sequence', {})
            expected_train_ids = set(train_sequences['sequence_id'])
            partial_features = {
                sequence_id: feature.float().cpu()
                for sequence_id, feature in candidate_features.items()
                if sequence_id in expected_train_ids and tuple(feature.shape) == (NUM_FRAMES, FEATURE_DIM)
            }
    missing_train_sequences = train_sequences.loc[~train_sequences['sequence_id'].isin(partial_features)].copy()
    print(f'Resumable train feature cache: {len(partial_features)} / {len(train_sequences)} sequences already available.')

    if len(missing_train_sequences):
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        backbone = models.resnet18(weights=weights)
        encoder = nn.Sequential(*list(backbone.children())[:-1]).to(device).eval()
        for parameter in encoder.parameters():
            parameter.requires_grad_(False)

        missing_dataset = TrainSequenceFrameDataset(missing_train_sequences, train_frame_groups)
        missing_loader = DataLoader(
            missing_dataset, batch_size=FRAME_BATCH_SIZE, shuffle=False,
            num_workers=0, pin_memory=(device.type == 'cuda'),
        )
        batches_since_save = 0
        with torch.inference_mode():
            for images, _, batch_sequence_ids in tqdm(missing_loader, desc='Extracting new train ResNet18 features'):
                batch_size, time_steps, channels, height, width = images.shape
                flattened_images = images.reshape(batch_size * time_steps, channels, height, width).to(device)
                batch_features = encoder(flattened_images).flatten(1).reshape(batch_size, time_steps, FEATURE_DIM).cpu()
                for sequence_id, sequence_features in zip(batch_sequence_ids, batch_features):
                    partial_features[sequence_id] = sequence_features
                batches_since_save += 1
                if batches_since_save >= 25:
                    temporary_path = TRAIN_FEATURE_PARTIAL_PATH.with_suffix('.tmp')
                    torch.save({'preprocessing_version': PREPROCESSING_VERSION, 'features_by_sequence': partial_features}, temporary_path)
                    temporary_path.replace(TRAIN_FEATURE_PARTIAL_PATH)
                    batches_since_save = 0

        temporary_path = TRAIN_FEATURE_PARTIAL_PATH.with_suffix('.tmp')
        torch.save({'preprocessing_version': PREPROCESSING_VERSION, 'features_by_sequence': partial_features}, temporary_path)
        temporary_path.replace(TRAIN_FEATURE_PARTIAL_PATH)

    assert set(partial_features) == set(train_sequences['sequence_id'])
    feature_by_sequence = dict(partial_features)
    for _, validation_row in validation_sequences.iterrows():
        base_sequence_id = base_sequence_id_by_video[validation_row.video_id]
        feature_by_sequence[validation_row.sequence_id] = base_feature_by_sequence[base_sequence_id]

    assert set(feature_by_sequence) == set(expected_sequence_ids)
    feature_payload = {
        'features': torch.stack([feature_by_sequence[sequence_id] for sequence_id in expected_sequence_ids]),
        'labels': torch.as_tensor(sequence_manifest['label'].to_numpy(), dtype=torch.long),
        'sequence_ids': expected_sequence_ids,
        'video_ids': sequence_manifest['video_id'].tolist(),
        'splits': sequence_manifest['split'].tolist(),
        'preprocessing_version': PREPROCESSING_VERSION,
        'encoder': 'ResNet18_Weights.IMAGENET1K_V1 (frozen)',
        'train_feature_source': 'new multiposition RGB cache',
        'validation_feature_source': 'reused frozen V2-W2 cache',
    }
    torch.save(feature_payload, FEATURE_CACHE_PATH)

features = feature_payload['features'].float()
labels = feature_payload['labels'].long()
assert features.shape == (1560, NUM_FRAMES, FEATURE_DIM)
assert feature_payload['sequence_ids'] == expected_sequence_ids
assert labels.tolist() == sequence_manifest['label'].tolist()
print({
    'feature_cache': str(FEATURE_CACHE_PATH),
    'reused_complete_feature_cache': reused_complete_feature_cache,
    'feature_shape': tuple(features.shape),
})

Resumable train feature cache: 0 / 1440 sequences already available.


Extracting new train ResNet18 features: 100%|██████████| 720/720 [36:31<00:00,  3.04s/it]  
C:\Users\User\AppData\Local\Temp\ipykernel_3880\3517130078.py:80: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  'labels': torch.as_tensor(sequence_manifest['label'].to_numpy(), dtype=torch.long),


{'feature_cache': 'P:\\NexarCollisionData\\processed_v2\\resnet18_imagenet_features_v2_multipos_16x224x320.pt', 'reused_complete_feature_cache': False, 'feature_shape': (1560, 16, 512)}


In [5]:
class SequenceFeatureDataset(Dataset):
    def __init__(self, features: torch.Tensor, labels: torch.Tensor, indices: np.ndarray):
        self.features = features
        self.labels = labels
        self.indices = torch.as_tensor(indices, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, index: int):
        source_index = self.indices[index]
        return self.features[source_index], self.labels[source_index], int(source_index)

class ResNet18MeanMaxPoolingHead(nn.Module):
    def __init__(self, feature_dim: int = FEATURE_DIM, dropout: float = 0.35):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim * 2),
            nn.Dropout(dropout),
            nn.Linear(feature_dim * 2, 1),
        )

    def forward(self, sequence_features: torch.Tensor, frame_mask: torch.Tensor | None = None) -> torch.Tensor:
        if frame_mask is None:
            mean_features = sequence_features.mean(dim=1)
            max_features = sequence_features.max(dim=1).values
        else:
            valid = frame_mask.bool()
            mean_weights = valid.float().unsqueeze(-1)
            mean_features = (sequence_features * mean_weights).sum(dim=1) / mean_weights.sum(dim=1).clamp_min(1.0)
            masked_features = sequence_features.masked_fill(~valid.unsqueeze(-1), float('-inf'))
            max_features = masked_features.max(dim=1).values
        return self.classifier(torch.cat([mean_features, max_features], dim=1)).squeeze(1)

train_indices = np.flatnonzero(sequence_manifest['split'].eq('train').to_numpy())
validation_indices = np.flatnonzero(sequence_manifest['split'].eq('validation').to_numpy())
assert len(train_indices) == 1440 and len(validation_indices) == 120
train_loader = DataLoader(SequenceFeatureDataset(features, labels, train_indices), batch_size=HEAD_BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(SequenceFeatureDataset(features, labels, validation_indices), batch_size=HEAD_BATCH_SIZE, shuffle=False, num_workers=0)

model = ResNet18MeanMaxPoolingHead().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()
print(model)

ResNet18MeanMaxPoolingHead(
  (classifier): Sequential(
    (0): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Dropout(p=0.35, inplace=False)
    (2): Linear(in_features=1024, out_features=1, bias=True)
  )
)


In [6]:
def binary_metrics(y_true: np.ndarray, probabilities: np.ndarray, threshold: float) -> dict:
    predictions = (probabilities >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, predictions)),
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'recall': float(recall_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, probabilities)),
        'pr_auc': float(average_precision_score(y_true, probabilities)),
        'confusion_matrix': confusion_matrix(y_true, predictions).tolist(),
    }

def evaluate(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    labels_out, probabilities_out, indices_out = [], [], []
    with torch.inference_mode():
        for batch_features, batch_labels, batch_indices in loader:
            logits = model(batch_features.to(device))
            labels_out.append(batch_labels.numpy())
            probabilities_out.append(torch.sigmoid(logits).cpu().numpy())
            indices_out.append(batch_indices.numpy())
    return np.concatenate(labels_out), np.concatenate(probabilities_out), np.concatenate(indices_out)

best_pr_auc = -np.inf
best_epoch = 0
epochs_without_improvement = 0
history = []
best_model_path = MODEL_DIR / f'{MODEL_NAME}_best.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    for batch_features, batch_labels, _ in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.float().to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(batch_features), batch_labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(batch_labels)

    validation_labels, validation_probabilities, _ = evaluate(model, validation_loader)
    metrics_at_05 = binary_metrics(validation_labels, validation_probabilities, threshold=0.5)
    record = {
        'epoch': epoch,
        'train_loss': loss_sum / len(train_loader.dataset),
        'validation_accuracy_at_0_5': metrics_at_05['accuracy'],
        'validation_f1_at_0_5': metrics_at_05['f1'],
        'validation_recall_at_0_5': metrics_at_05['recall'],
        'validation_pr_auc': metrics_at_05['pr_auc'],
        'validation_roc_auc': metrics_at_05['roc_auc'],
    }
    history.append(record)
    print(record)

    # Checkpoint selection is threshold-independent; threshold tuning happens only after training.
    if record['validation_pr_auc'] > best_pr_auc:
        best_pr_auc = record['validation_pr_auc']
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'model_state_dict': model.state_dict(), 'epoch': epoch,
            'validation_pr_auc': best_pr_auc, 'model_name': MODEL_NAME,
            'feature_dim': FEATURE_DIM, 'num_frames': NUM_FRAMES,
            'encoder': 'ResNet18_Weights.IMAGENET1K_V1 (frozen)',
            'training_data': 'V2 multi-position train sequences (event at approximately 1.0, 2.5, 4.0 seconds)',
        }, best_model_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping at epoch {epoch}; best epoch: {best_epoch}')
            break

history_path = MODEL_DIR / f'{MODEL_NAME}_training_history.csv'
pd.DataFrame(history).to_csv(history_path, index=False)
print(f'Best epoch by validation PR-AUC: {best_epoch}, PR-AUC={best_pr_auc:.4f}')

{'epoch': 1, 'train_loss': 0.6440422177314759, 'validation_accuracy_at_0_5': 0.6916666666666667, 'validation_f1_at_0_5': 0.6542056074766355, 'validation_recall_at_0_5': 0.5833333333333334, 'validation_pr_auc': 0.692106238719215, 'validation_roc_auc': 0.7497222222222223}
{'epoch': 2, 'train_loss': 0.5937106000052558, 'validation_accuracy_at_0_5': 0.75, 'validation_f1_at_0_5': 0.75, 'validation_recall_at_0_5': 0.75, 'validation_pr_auc': 0.7005737006221404, 'validation_roc_auc': 0.7619444444444445}
{'epoch': 3, 'train_loss': 0.5518837889035543, 'validation_accuracy_at_0_5': 0.75, 'validation_f1_at_0_5': 0.7580645161290323, 'validation_recall_at_0_5': 0.7833333333333333, 'validation_pr_auc': 0.7079992059160825, 'validation_roc_auc': 0.7694444444444445}
{'epoch': 4, 'train_loss': 0.5431149058871799, 'validation_accuracy_at_0_5': 0.7666666666666667, 'validation_f1_at_0_5': 0.7627118644067796, 'validation_recall_at_0_5': 0.75, 'validation_pr_auc': 0.7034248584614441, 'validation_roc_auc': 0.7

In [7]:
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
validation_labels, validation_probabilities, validation_indices_out = evaluate(model, validation_loader)

threshold_table = pd.DataFrame([
    binary_metrics(validation_labels, validation_probabilities, float(threshold))
    for threshold in np.round(np.arange(0.10, 0.901, 0.01), 2)
])
safe_thresholds = threshold_table.loc[threshold_table['recall'].ge(MINIMUM_ACCIDENT_RECALL)]
selection_pool = safe_thresholds if len(safe_thresholds) else threshold_table
selected_row = selection_pool.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0]
selected_threshold = float(selected_row['threshold'])
metrics_at_05 = binary_metrics(validation_labels, validation_probabilities, 0.5)
metrics_at_selected_threshold = binary_metrics(validation_labels, validation_probabilities, selected_threshold)

prediction_table = sequence_manifest.iloc[validation_indices_out].copy().reset_index(drop=True)
prediction_table['positive_probability'] = validation_probabilities
prediction_table['prediction_at_0_5'] = (validation_probabilities >= 0.5).astype(int)
prediction_table['prediction_at_selected_threshold'] = (validation_probabilities >= selected_threshold).astype(int)
prediction_table['selected_threshold'] = selected_threshold

predictions_path = MODEL_DIR / f'{MODEL_NAME}_clip_validation_predictions.csv'
threshold_path = MODEL_DIR / f'{MODEL_NAME}_threshold_curve.csv'
metrics_path = MODEL_DIR / f'{MODEL_NAME}_clip_metrics.json'
prediction_table.to_csv(predictions_path, index=False)
threshold_table.to_csv(threshold_path, index=False)

metrics_payload = {
    'model_name': MODEL_NAME,
    'evaluation_scope': 'clip-level validation on fixed V2-W2 sequences; full-MP4 sliding-window inference is required before final selection',
    'checkpoint_selection_metric': 'validation PR-AUC; decision threshold not tuned during training',
    'best_epoch': int(checkpoint['epoch']),
    'minimum_accident_recall_for_threshold_selection': MINIMUM_ACCIDENT_RECALL,
    'selected_threshold_by_validation_f1_under_recall_constraint': selected_threshold,
    'metrics_at_threshold_0_5': metrics_at_05,
    'metrics_at_selected_threshold': metrics_at_selected_threshold,
    'encoder': 'ResNet18 ImageNet frozen',
    'temporal_aggregator': 'mean-max pooling over 16 frame features',
    'training_sequences': 1440,
    'validation_sequences': 120,
    'input_shape_per_sequence': [NUM_FRAMES, 3, 224, 320],
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

figure, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(validation_labels, (validation_probabilities >= 0.5).astype(int), ax=axes[0], colorbar=False)
axes[0].set_title('Clip validation, threshold = 0.50')
ConfusionMatrixDisplay.from_predictions(validation_labels, (validation_probabilities >= selected_threshold).astype(int), ax=axes[1], colorbar=False)
axes[1].set_title(f'Clip validation, threshold = {selected_threshold:.2f}')
figure.tight_layout()
confusion_path = MODEL_DIR / f'{MODEL_NAME}_clip_confusion_matrices.png'
figure.savefig(confusion_path, dpi=160)
plt.close(figure)

comparison_rows = []
if BASELINE_METRICS_PATH.exists():
    baseline = json.loads(BASELINE_METRICS_PATH.read_text(encoding='utf-8'))
    baseline_metrics = baseline['metrics_at_selected_threshold']
    comparison_rows.append({
        'model': 'A2 frozen ResNet18 + mean-max (original V2 train)',
        'f1': baseline_metrics['f1'], 'recall': baseline_metrics['recall'],
        'precision': baseline_metrics['precision'], 'pr_auc': baseline_metrics['pr_auc'],
        'threshold': baseline['selected_threshold_by_validation_f1'],
    })
comparison_rows.append({
    'model': 'A2-MP frozen ResNet18 + mean-max (three train event positions)',
    'f1': metrics_at_selected_threshold['f1'], 'recall': metrics_at_selected_threshold['recall'],
    'precision': metrics_at_selected_threshold['precision'], 'pr_auc': metrics_at_selected_threshold['pr_auc'],
    'threshold': selected_threshold,
})
comparison_table = pd.DataFrame(comparison_rows)
comparison_path = MODEL_DIR / 'v2_multipos_clip_ablation_comparison.csv'
comparison_table.to_csv(comparison_path, index=False)

print('Clip metrics at threshold 0.50:')
print(metrics_at_05)
print('Clip metrics at the validation-selected threshold:')
print(metrics_at_selected_threshold)
display(comparison_table)
print(f'Model: {best_model_path}')
print(f'Feature cache: {FEATURE_CACHE_PATH}')
print(f'Comparison: {comparison_path}')

Clip metrics at threshold 0.50:
{'threshold': 0.5, 'accuracy': 0.7083333333333334, 'precision': 0.7659574468085106, 'recall': 0.6, 'f1': 0.6728971962616822, 'roc_auc': 0.7808333333333333, 'pr_auc': 0.7304503474487104, 'confusion_matrix': [[49, 11], [24, 36]]}
Clip metrics at the validation-selected threshold:
{'threshold': 0.3, 'accuracy': 0.7333333333333333, 'precision': 0.6891891891891891, 'recall': 0.85, 'f1': 0.7611940298507462, 'roc_auc': 0.7808333333333333, 'pr_auc': 0.7304503474487104, 'confusion_matrix': [[37, 23], [9, 51]]}


,model,f1,recall,precision,pr_auc,threshold
0,A2 frozen ResNet18 + mean-max (original V2 train),0.787402,0.833333,0.746269,0.742437,0.44
1,A2-MP frozen ResNet18 + mean-max (three train ...,0.761194,0.850000,0.689189,0.730450,0.30


Model: P:\NexarCollisionData\models_v2\resnet18_meanmax_pooling_frozen_multipos_best.pt
Feature cache: P:\NexarCollisionData\processed_v2\resnet18_imagenet_features_v2_multipos_16x224x320.pt
Comparison: P:\NexarCollisionData\models_v2\v2_multipos_clip_ablation_comparison.csv
